## Theory + Abstract Class

In [1]:
abstract class PaymentProcessor {
    // Abstract — must be implemented by subclasses
    protected abstract boolean validatePayment(double amount);
    protected abstract String processTransaction(double amount, String to);
    protected abstract String getProcessorName();
    
    // Concrete — shared by all
    public final void pay(double amount, String recipient) {
        System.out.println("\n[" + getProcessorName() + "] Processing payment...");
        if (!validatePayment(amount)) {
            System.out.println("Validation failed!");
            return;
        }
        String txId = processTransaction(amount, recipient);
        System.out.println("Payment SUCCESS | TxID: " + txId);
    }
}

class UPIProcessor extends PaymentProcessor {
    private String upiId;
    private double balance;
    
    UPIProcessor(String upiId, double balance) { this.upiId = upiId; this.balance = balance; }
    
    @Override
    protected boolean validatePayment(double amount) {
        boolean ok = balance >= amount;
        System.out.println("UPI balance check: " + (ok ? "OK" : "FAILED"));
        return ok;
    }
    
    @Override
    protected String processTransaction(double amount, String to) {
        balance -= amount;
        return "UPI-" + System.currentTimeMillis();
    }
    
    @Override protected String getProcessorName() { return "UPI (" + upiId + ")"; }
}

PaymentProcessor processor = new UPIProcessor("user@icici", 5000);
processor.pay(1500, "merchant@axis");
processor.pay(10000, "expensive@shop"); // fails


[UPI (user@icici)] Processing payment...
UPI balance check: OK
Payment SUCCESS | TxID: UPI-1781077873006

[UPI (user@icici)] Processing payment...
UPI balance check: FAILED
Validation failed!


## Template Method Pattern

In [2]:
// Template method pattern
abstract class ReportGenerator {
    // Template method — final prevents override
    public final void generate() {
        fetchData();
        processData();
        formatReport();
        saveReport();
        System.out.println(getReportType() + " report generated!\n");
    }
    
    protected abstract String getReportType();
    protected void fetchData() { System.out.println("Fetching data from DB..."); }
    protected abstract void processData();
    protected abstract void formatReport();
    protected void saveReport() { System.out.println("Saving to /reports/"); }
}

class SalesReport extends ReportGenerator {
    @Override protected String getReportType() { return "Sales"; }
    @Override protected void processData() { System.out.println("Aggregating sales by region..."); }
    @Override protected void formatReport() { System.out.println("Formatting as table with charts..."); }
}

class AuditReport extends ReportGenerator {
    @Override protected String getReportType() { return "Audit"; }
    @Override protected void processData() { System.out.println("Analyzing transaction logs..."); }
    @Override protected void formatReport() { System.out.println("Formatting as compliance document..."); }
}

new SalesReport().generate();
new AuditReport().generate();

Fetching data from DB...
Aggregating sales by region...
Formatting as table with charts...
Saving to /reports/
Sales report generated!

Fetching data from DB...
Analyzing transaction logs...
Formatting as compliance document...
Saving to /reports/
Audit report generated!



## Mini Challenge
Create abstract class `DataExporter` with template method `export()` that: validates → formats → writes → notifies. Subclasses implement `format()` and `write()` for CSV, JSON, and PDF.

In [3]:
import java.util.ArrayList;
import java.util.List;
import java.util.Map;

// 1. Abstract Class Defining the Template Method
abstract class DataExporter {
    private String fileName;

    public DataExporter(String fileName) {
        this.fileName = fileName;
    }

    public String getFileName() {
        return fileName;
    }

    // The Template Method - Defines the invariant skeleton of the algorithm
    public final void export(List<Map<String, String>> data) {
        validate(data);
        String formattedData = format(data);
        write(formattedData);
        notifyUser();
        System.out.println("--------------------------------------------------\n");
    }

    // Concrete step built into the template
    private void validate(List<Map<String, String>> data) {
        System.out.println("[" + fileName + "] Step 1: Validating data consistency...");
        if (data == null || data.isEmpty()) {
            throw new IllegalArgumentException("Data cannot be null or empty.");
        }
    }

    // Primitive operations to be implemented by subclasses
    protected abstract String format(List<Map<String, String>> data);
    protected abstract void write(String formattedData);

    // Concrete step built into the template
    private void notifyUser() {
        System.out.println("[" + fileName + "] Step 4: Notification sent! Export complete.");
    }
}

// 2. Subclass for CSV Export
class CSVExporter extends DataExporter {
    public CSVExporter(String fileName) {
        super(fileName);
    }

    @Override
    protected String format(List<Map<String, String>> data) {
        System.out.println("[" + getFileName() + "] Step 2: Formatting data to CSV rows...");
        // Simple simulation of CSV conversion
        StringBuilder csv = new StringBuilder("id,name,role\n");
        for (Map<String, String> row : data) {
            csv.append(row.get("id")).append(",")
               .append(row.get("name")).append(",")
               .append(row.get("role")).append("\n");
        }
        return csv.toString().trim();
    }

    @Override
    protected void write(String formattedData) {
        System.out.println("[" + getFileName() + "] Step 3: Writing CSV text to disk...");
        System.out.println("--- FILE CONTENT ---\n" + formattedData + "\n--------------------");
    }
}

// 3. Subclass for JSON Export
class JSONExporter extends DataExporter {
    public JSONExporter(String fileName) {
        super(fileName);
    }

    @Override
    protected String format(List<Map<String, String>> data) {
        System.out.println("[" + getFileName() + "] Step 2: Formatting data to a JSON Array...");
        return "[\n  { \"id\": 1, \"name\": \"Alice\" },\n  { \"id\": 2, \"name\": \"Bob\" }\n]";
    }

    @Override
    protected void write(String formattedData) {
        System.out.println("[" + getFileName() + "] Step 3: Writing raw JSON string to file system...");
        System.out.println("--- FILE CONTENT ---\n" + formattedData + "\n--------------------");
    }
}

// 4. Subclass for PDF Export
class PDFExporter extends DataExporter {
    public PDFExporter(String fileName) {
        super(fileName);
    }

    @Override
    protected String format(List<Map<String, String>> data) {
        System.out.println("[" + getFileName() + "] Step 2: Generating PDF binary stream layout...");
        return "%PDF-1.4 [Document Matrix Data]";
    }

    @Override
    protected void write(String formattedData) {
        System.out.println("[" + getFileName() + "] Step 3: Compiling stream and saving PDF file...");
        System.out.println("--- FILE CONTENT ---\n" + formattedData + "\n--------------------");
    }
}

// 5. Execution Cell
// Mock Data
List<Map<String, String>> sampleData = new ArrayList<>();
sampleData.add(Map.of("id", "1", "name", "Alice", "role", "Dev"));
sampleData.add(Map.of("id", "2", "name", "Bob", "role", "Ops"));

// Polymorphic list of exporters running the template method
List<DataExporter> exporters = List.of(
    new CSVExporter("report.csv"),
    new JSONExporter("data.json"),
    new PDFExporter("analytics.pdf")
);

System.out.println("=== Starting Template Method Data Export ===\n");
for (DataExporter exporter : exporters) {
    exporter.export(sampleData);
}

=== Starting Template Method Data Export ===

[report.csv] Step 1: Validating data consistency...
[report.csv] Step 2: Formatting data to CSV rows...
[report.csv] Step 3: Writing CSV text to disk...
--- FILE CONTENT ---
id,name,role
1,Alice,Dev
2,Bob,Ops
--------------------
[report.csv] Step 4: Notification sent! Export complete.
--------------------------------------------------

[data.json] Step 1: Validating data consistency...
[data.json] Step 2: Formatting data to a JSON Array...
[data.json] Step 3: Writing raw JSON string to file system...
--- FILE CONTENT ---
[
  { "id": 1, "name": "Alice" },
  { "id": 2, "name": "Bob" }
]
--------------------
[data.json] Step 4: Notification sent! Export complete.
--------------------------------------------------

[analytics.pdf] Step 1: Validating data consistency...
[analytics.pdf] Step 2: Generating PDF binary stream layout...
[analytics.pdf] Step 3: Compiling stream and saving PDF file...
--- FILE CONTENT ---
%PDF-1.4 [Document Matrix Dat